# Audit v3: error bars, the screen at large N, and the converse relabelled

Two things `finalize_audit.py` cannot do without a GPU, and one relabelling.

| | What | Why |
|---|---|---|
| A | Re-run the screen at `N = 2048` | Every number in the v2 ladder used `\|E\|` calibrated at `N = 256`. More samples give replay noise more chances to exceed `eta`, so `\|E\|` must be re-measured before the construction rungs can be quoted at 2048. |
| B | Repeat each pool mode over several tampers | Every v2 number is a single instance. |
| C | Converse with column counts | v2's "head profile" measures invisible changes *anywhere*, including `delta = 0` on untouched rows — a uniform converse over all tampers, not a per-tamper residual. Reporting the column counts makes that visible instead of confusing. |

Reuses the v2 caches under `repair_runs/audit_v2`, and verifies they belong to the same texts
before trusting them. First run also caches the head, so later runs skip the model entirely.

Budget: a few minutes once the caches are warm.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import gc, json, math, time

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm

import interval_audit as ia


@dataclass(frozen=True)
class Config:
    model_id: str = "meta-llama/Meta-Llama-3.1-8B-Instruct"
    hf_cache_dir: str = "/scratch/bbjr/skarmakar/huggingface"
    model_revision: str | None = None
    dataset_revision: str | None = None
    cache_dir: str = "repair_runs/audit_v2"        # reuse the v2 features and codes
    out_dir: str = "repair_runs/audit_v3"
    pilot_seed: int = 17
    sparsity: int = 64
    output_bits: int = 8
    fixed_logit_range: float = 64.0
    rho_multiple: float = 10.0
    tamper_fraction_of_bound: float = 0.9
    constraint_slack: float = 1e-4
    head_batch: int = 32
    interval_chunk: int = 1024
    verify_cap: int = 200_000
    estimate_draws: int = 2048
    pool_request: int = 8192
    cal_contexts: int = 96
    N_MAX: int = 2048
    eta: float = 1e-5                              # calibrated in v2
    n_reference: tuple = (256, 2048)               # report at both
    trials: tuple = (1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007)  # 1000 reproduces v2
    pool_modes: tuple = ("salient", "uniform", "salient_rows_uniform_cols",
                         "uniform_rows_salient_cols")
    converse_rows: int = 512

CFG = Config()
assert torch.cuda.is_available(), "A CUDA GPU is required."
DEVICE = torch.device("cuda")
OUT = Path(CFG.out_dir); OUT.mkdir(parents=True, exist_ok=True)
CACHE = Path(CFG.cache_dir)
R = float(CFG.fixed_logit_range); WIDTH = 2 * R / (2 ** CFG.output_bits)
VALUES, LABELS = ia.bf16_table()
REPORT = {}
print({"cache": str(CACHE), "out": str(OUT), "eta": CFG.eta})

## Setup, from cache

The pool texts are regenerated by the same deterministic rule as v2. The caches are keyed only by
length, so a one-sample code check confirms they belong to these texts before anything is
trusted.

In [ ]:
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", revision=CFG.dataset_revision)
usable = lambda sp: [x["text"].strip() for x in ds[sp] if len(x["text"].strip()) >= 80]
held_out = list(dict.fromkeys(usable("validation") + usable("test")))
_rng = np.random.default_rng(CFG.pilot_seed + 2)
pool_n = min(CFG.pool_request, len(held_out))
pool_texts = [held_out[i] for i in _rng.choice(len(held_out), pool_n, replace=False)]
cal_texts = [usable("train")[i] for i in np.random.default_rng(CFG.pilot_seed).choice(
    len(usable("train")), CFG.cal_contexts, replace=False)]

H_POOL = torch.load(CACHE / f"hidden_pool_{pool_n}.pt", map_location="cpu", weights_only=True)
H_CAL = torch.load(CACHE / f"hidden_cal_{CFG.cal_contexts}.pt", map_location="cpu", weights_only=True)
CODES_POOL = torch.load(CACHE / f"codes_pool_{pool_n}_y{CFG.output_bits}.pt",
                        map_location="cpu", weights_only=True)
print({"pool": pool_n, "H_POOL": tuple(H_POOL.shape), "CODES_POOL": tuple(CODES_POOL.shape)})

In [ ]:
head_path = CACHE / "W_original_bf16.pt"
if head_path.exists():
    W_ORIGINAL = torch.load(head_path, map_location="cpu", weights_only=True).to(DEVICE)
    print("head loaded from cache; model not needed")
else:
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(
        CFG.model_id, cache_dir=CFG.hf_cache_dir, revision=CFG.model_revision,
        torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
        attn_implementation="sdpa").eval().to(DEVICE)
    assert not bool(getattr(model.config, "tie_word_embeddings", True))
    W_ORIGINAL = model.lm_head.weight.detach().clone()
    torch.save(W_ORIGINAL.cpu(), head_path)
    del model; gc.collect(); torch.cuda.empty_cache()
    W_ORIGINAL = W_ORIGINAL.to(DEVICE)
    print("head extracted and cached")

V, D = W_ORIGINAL.shape
weight_rms = float(W_ORIGINAL.float().square().mean().sqrt())
rho = CFG.rho_multiple * weight_rms
tamper_step = CFG.tamper_fraction_of_bound * rho
row_scale = W_ORIGINAL.float().abs().amax(1).clamp_min(1e-12) / 127.

def qcode(z):
    if bool(((z < -R) | (z >= R)).any()):
        raise RuntimeError("Public logit range saturated.")
    return torch.floor((z + R) / WIDTH).to(torch.uint8)

# Cache guard: the caches are keyed by length only, so confirm they are for THESE texts.
probe = qcode(H_POOL[:4].to(DEVICE, torch.float32) @ W_ORIGINAL.float().T).cpu()
assert torch.equal(probe, CODES_POOL[:4]), \
    "Cached codes do not match this head and these texts. Delete the cache and rerun v2."
print({"vocab": V, "hidden": D, "rho": rho, "cache_guard": "passed"})

In [ ]:
rng = np.random.default_rng(CFG.pilot_seed + 3)
order = rng.permutation(len(H_POOL))[:CFG.N_MAX]          # same rule as v2
retained_H = H_POOL[order].to(DEVICE, torch.float32)
retained_codes = CODES_POOL[torch.as_tensor(order)]

with torch.inference_mode():
    hcal = H_CAL.to(DEVICE, torch.float32)
    salient_rows = torch.topk(torch.softmax(hcal @ W_ORIGINAL.float().T, -1).mean(0),
                              min(256, V)).indices.cpu().numpy()
    salient_cols = torch.topk(H_CAL.square().mean(0), min(512, D)).indices.cpu().numpy()
    del hcal; torch.cuda.empty_cache()
_r = np.random.default_rng(CFG.pilot_seed + 11)
uniform_rows = _r.choice(V, len(salient_rows), replace=False)
uniform_cols = _r.choice(D, len(salient_cols), replace=False)
POOLS = {"salient": (salient_rows, salient_cols), "uniform": (uniform_rows, uniform_cols),
         "salient_rows_uniform_cols": (salient_rows, uniform_cols),
         "uniform_rows_salient_cols": (uniform_rows, salient_cols)}
MODE_INDEX = {m: i for i, m in enumerate(sorted(POOLS))}


def make_tamper(mode, W, trial):
    """EVALUATOR ONLY. Same construction and seeding rule as v2."""
    rows_pool, cols_pool = POOLS[mode]
    g = np.random.default_rng(CFG.pilot_seed + 10_000 * trial + 137 * MODE_INDEX[mode])
    rows = g.choice(rows_pool, CFG.sparsity, replace=False).astype(np.int64)
    cols = g.choice(cols_pool, CFG.sparsity, replace=True).astype(np.int64)
    signs = g.choice([-1, 1], CFG.sparsity).astype(np.int64)
    rr = torch.as_tensor(rows, device=DEVICE); cc = torch.as_tensor(cols, device=DEVICE)
    scale = row_scale[rr]
    old_signed = torch.round(W[rr, cc].float() / scale).clamp(-127, 127).to(torch.int64)
    step = torch.round(torch.full_like(scale, tamper_step) / scale).clamp_min(1).to(torch.int64)
    step = torch.minimum(step, torch.floor(torch.full_like(scale, rho) / scale).to(torch.int64))
    sign = torch.as_tensor(signs, device=DEVICE)
    proposed = (old_signed + sign * step).clamp(-127, 127)
    proposed = torch.where(proposed == old_signed,
                           (old_signed - sign * step).clamp(-127, 127), proposed)
    if bool((proposed == old_signed).any()):
        raise RuntimeError(f"[{mode}/{trial}] INT8 reference change vanished.")
    old = W[rr, cc].clone()
    new = (old.float() + (proposed - old_signed).float() * scale).to(torch.bfloat16)
    if bool((new == old).any()) or float((new.float() - old.float()).abs().max()) > rho:
        raise RuntimeError(f"[{mode}/{trial}] BF16 tamper violates the contract.")
    return {"rows": rows, "cols": cols, "old": old, "new": new, "rr": rr, "cc": cc}


def audit(W_dam, rows, hN, codesN, rng):
    """Per-row candidate counts and column counts. Exact where affordable, else estimated."""
    recs = []
    for row in rows:
        z = W_dam[int(row)].float()
        cr = codesN[:, int(row)].to(DEVICE)
        lo, hi, feas = ia.row_intervals(z, hN, cr, R, WIDTH, CFG.constraint_slack, rho,
                                        chunk=CFG.interval_chunk)
        cnt, left, right = ia.count_candidates(z, lo, hi, feas, VALUES)
        exact, per_col = ia.verify_row(z, hN, cr, left, right, cnt, VALUES, R, WIDTH,
                                       CFG.verify_cap)
        if exact is None:
            e = ia.estimate_row(z, hN, cr, left, right, cnt, VALUES, R, WIDTH,
                                n_samples=CFG.estimate_draws, rng=rng)
            recs.append({"row": int(row), "method": "estimate", "candidates": e["estimate"],
                         "lower": e["lower"], "upper": e["upper"],
                         "feasible_columns": int((cnt > 0).sum()), "counts": cnt})
        else:
            recs.append({"row": int(row), "method": "exact", "candidates": float(exact),
                         "lower": float(exact), "upper": float(exact),
                         "feasible_columns": int((per_col > 0).sum()), "counts": per_col})
    return recs

## A — the screen at each reference N

`|E|` is the only quantity the construction rungs depend on that the offline script cannot
derive. Recall must stay 1.0; watch whether false positives reappear at the larger N.

In [ ]:
screen = []
for n in CFG.n_reference:
    for mode in CFG.pool_modes:
        W = W_ORIGINAL.clone()
        t = make_tamper(mode, W, CFG.trials[0])
        with torch.no_grad():
            W[t["rr"], t["cc"]] = t["new"]
        flagged, _ = ia.screen_rows(retained_H[:n], retained_codes[:n], W.float(),
                                    R, WIDTH, CFG.eta, chunk=CFG.head_batch)
        truth = set(int(r) for r in t["rows"]); got = set(int(r) for r in flagged)
        screen.append({"N": n, "mode": mode, "flagged": len(got),
                       "recall": len(got & truth) / len(truth),
                       "false_positives": len(got - truth)})
        del W; gc.collect(); torch.cuda.empty_cache()
SC = pd.DataFrame(screen); SC.to_csv(OUT / "screen_by_N.csv", index=False)
REPORT["A_screen_by_N"] = screen
print(SC.to_string(index=False))
E_AT = {n: int(SC[SC["N"] == n]["flagged"].max()) for n in CFG.n_reference}
print("\n|E| to quote per N (worst case over modes):", E_AT)
if (SC["recall"] < 1.0).any():
    print("WARNING: recall below 1.0 somewhere — the construction rungs do not apply there.")

## B — trials

Eight tampers per pool mode. Trial 1000 reproduces the v2 instance, so its row should match the
v2 report exactly.

In [ ]:
rows_out = []
for mode in CFG.pool_modes:
    for trial in tqdm(CFG.trials, desc=mode, leave=False):
        W = W_ORIGINAL.clone()
        t = make_tamper(mode, W, trial)
        with torch.no_grad():
            W[t["rr"], t["cc"]] = t["new"]
        for n in CFG.n_reference:
            flagged, _ = ia.screen_rows(retained_H[:n], retained_codes[:n], W.float(),
                                        R, WIDTH, CFG.eta, chunk=CFG.head_batch)
            truth = set(int(r) for r in t["rows"])
            recs = audit(W, np.array(sorted(truth)), retained_H[:n], retained_codes[:n],
                         np.random.default_rng(CFG.pilot_seed + trial + n))
            surv = all(r["counts"][int(t["cols"][int(np.flatnonzero(t["rows"] == r["row"])[0])])] > 0
                       for r in recs)
            cand = np.array([r["candidates"] for r in recs])
            rows_out.append({
                "mode": mode, "trial": trial, "N": n,
                "flagged": int(len(flagged)),
                "recall": len(set(int(r) for r in flagged) & truth) / len(truth),
                "false_positives": int(len(set(int(r) for r in flagged) - truth)),
                "columns_pinned": int(sum(r["feasible_columns"] == 1 for r in recs)),
                "median_candidates": float(np.median(cand)),
                "residual_bits": float(np.log2(np.maximum(cand, 1)).sum()),
                "rows_exact": int(sum(r["method"] == "exact" for r in recs)),
                "true_coordinate_survives": bool(surv)})
        del W; gc.collect(); torch.cuda.empty_cache()

TR = pd.DataFrame(rows_out); TR.to_csv(OUT / "trials.csv", index=False)
agg = TR.groupby(["mode", "N"]).agg(
    trials=("trial", "count"), recall=("recall", "mean"),
    flagged_mean=("flagged", "mean"), false_pos_mean=("false_positives", "mean"),
    cols_pinned_mean=("columns_pinned", "mean"),
    residual_mean=("residual_bits", "mean"), residual_std=("residual_bits", "std"),
    median_cand=("median_candidates", "mean"),
    all_survive=("true_coordinate_survives", "all")).reset_index()
agg.to_csv(OUT / "trials_summary.csv", index=False)
REPORT["B_trials"] = agg.to_dict(orient="records")
print(agg.to_string(index=False))
assert bool(TR["true_coordinate_survives"].all()), "A true original was excluded — check the slack."

## C — the converse, with column counts

Untouched rows: nothing needs explaining, so `delta = 0` is feasible for every column and none is
eliminated. `feasible_columns` here should be near `D`, which is exactly why this number is far
larger than the per-tamper residual and why it answers a different question — what any encoder
must carry to cover *every* legal tamper, including the invisible ones.

In [ ]:
conv = []
g = np.random.default_rng(CFG.pilot_seed + 7)
sample_rows = g.choice(V, min(CFG.converse_rows, V), replace=False)
for n in CFG.n_reference:
    recs = audit(W_ORIGINAL, sample_rows, retained_H[:n], retained_codes[:n],
                 np.random.default_rng(CFG.pilot_seed + 7))
    lo_bits = np.log2(np.maximum([r["lower"] for r in recs], 1))
    conv.append({"N": n, "rows_sampled": len(recs),
                 "fraction_exact": float(np.mean([r["method"] == "exact" for r in recs])),
                 "median_feasible_columns": float(np.median([r["feasible_columns"] for r in recs])),
                 "hidden_size_D": D,
                 "median_bits_per_row": float(np.median(
                     np.log2(np.maximum([r["candidates"] for r in recs], 1)))),
                 "median_bits_lower_bound": float(np.median(lo_bits)),
                 "uniform_converse_bits": float(CFG.sparsity * np.median(lo_bits))})
CV = pd.DataFrame(conv); CV.to_csv(OUT / "uniform_converse.csv", index=False)
REPORT["C_uniform_converse"] = conv
print(CV.to_string(index=False))
print("\nRead as: any encoder covering every legal s-sparse tamper needs at least the last "
      "column; the per-tamper residual in section B is what remains for a tamper that happened.")

In [ ]:
base = 2 * CFG.sparsity * 31
final = []
for n in CFG.n_reference:
    for mode in CFG.pool_modes:
        a = agg[(agg["mode"] == mode) & (agg["N"] == n)].iloc[0]
        e = int(round(a["flagged_mean"]))
        final.append({"N": n, "mode": mode, "|E|": e,
                      "construction_bits": e * 17,
                      "residual_bits": round(a["residual_mean"]),
                      "residual_sd": round(a["residual_std"], 1) if a["trials"] > 1 else 0.0,
                      "construction_vs_free": f"{100 * (1 - e * 17 / base):.1f}%",
                      "residual_vs_free": f"{100 * (1 - a['residual_mean'] / base):.1f}%"})
FIN = pd.DataFrame(final); FIN.to_csv(OUT / "final_table.csv", index=False)
REPORT["D_final_table"] = final
REPORT["config"] = {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.__dict__.items()}
REPORT["provenance"] = {"torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
                        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S")}
(OUT / "audit_v3_report.json").write_text(json.dumps(REPORT, indent=2, default=float))
print(FIN.to_string(index=False))
print(f"\ndata-free baseline {base} bits; wrote {OUT / 'audit_v3_report.json'}")

## Then run the offline half

```
python finalize_audit.py --dir repair_runs/audit_v2 --at-n 2048 --flagged <|E| at 2048 above>
```

That gives the corrected per-row slope and the residual-information curve. Together with
`final_table.csv` the results section is complete: **stop measuring and write.**